# 03 · Agent RFT: Python-tool reasoning + sandboxed coding (generation + tools baseline eval)

The model calls a tool, the tool is **actually executed** in a sandbox, the result goes back to the model and the model
continues (multi-turn).

**Two kinds of tasks:**
- **Math/physics with a Python tool:** the model computes and checks with the `run_python` tool and gives the answer in `\boxed{}`.
- **Agentic coding (KodCode):** the model writes `solution.py` with the `read_file`, `write_file` and `run_tests` tools,
  runs the tests and fixes failures. The test file is restored to its original content before grading, so editing the tests does not help.

**Sections:**
- **A) Baseline eval (with tools):** MATH-500, AIME and GPQA physics with the Python tool, plus a **held-out set of 100 agentic coding tasks**.
- **B) RFT generation:** the **shortest** episodes that pass the tests / reach the right answer become training data.
  Training happens in `04_sft`, together with the round-1 data from `02` (rft2).

**Runtime and cost:**
- Runtime: **A100 80GB** (≈6.8 CU/hour).
- A takes ~1 hour, B at most `MAX_GEN_HOURS`. **~30–35 CU** in total.
- Resumable (`episodes.jsonl` on Drive).

**Before you start:** copy the repo's current `eval/` folder to Drive (`run_eval.py` and `agent.py`).
`02` must have run first: its `generations.jsonl` provides the hard problems and is excluded from the coding pool.

In [ ]:
!pip install -q -U vllm pytest

In [ ]:
import vllm, torch
print(vllm.__version__, torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

## Settings

In [ ]:
MODEL = "Qwen/Qwen3.6-35B-A3B-FP8"
MAX_MODEL_LEN = 20480
MAX_NUM_SEQS = 512
CU_PER_HOUR = 6.77
SEED = 3407

DRIVE_ROOT = "/content/drive/MyDrive/colab-ai"
EVAL_DIR = f"{DRIVE_ROOT}/eval"
ROUND1_GEN = f"{DRIVE_ROOT}/rft/round1/generations.jsonl"
OUT_DIR = f"{DRIVE_ROOT}/rft/agent1"

MAX_TURNS = {"python": 6, "code": 10}

# A) Baseline eval (with tools); same budget as the no-tools baseline: at most 16k generated tokens, 20k context
RUN_EVAL = True
EVAL_LABEL = "base-tools16k-colab-fp8"
EVAL_SETS = ["math500", "aime2025", "gpqa_physics"]
AGENT_EVAL_N = 100
EVAL_SAMPLES = 2
EVAL_TEMPERATURE = 0.6
EVAL_CONTEXT, EVAL_GEN_TOKENS = 20480, 16384
EVAL_TURN_TOKENS = 16384           # same thinking budget per turn as the no-tools eval (8192 cut AIME/MATH off in the first turn)

# B) RFT generation; training accepts examples up to 10240 tokens, so the episode context is capped at 12k
POOL = {"py_hard": 300, "py_new": 200, "code": 400}   # py_hard: math/physics problems solved ≤2/4 in round 1
N_SAMPLES = 4
GEN_TEMPERATURE = 0.8
GEN_CONTEXT, GEN_GEN_TOKENS = 12288, 12288
GEN_TURN_TOKENS = 8192             # deliberately short during generation: tool use instead of long one-shot thinking
CHUNK = 300                        # tasks; × N_SAMPLES episodes run concurrently
MAX_GEN_HOURS = 4.0

In [ ]:
import os, sys, json, glob, random, time, hashlib, importlib, collections
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")
login(token=userdata.get("HF_TOKEN"))
os.makedirs(OUT_DIR, exist_ok=True)
for f in ["run_eval.py", "agent.py"]:
    assert os.path.exists(f"{EVAL_DIR}/{f}"), f"{EVAL_DIR}/{f} not found: copy the repo's current eval/ folder to Drive"
sys.path.insert(0, EVAL_DIR)
sys.path_importer_cache.pop(EVAL_DIR, None)
importlib.invalidate_caches()
import run_eval as R
import agent as A


def uid(prefix, text):
    return f"{prefix}-{hashlib.md5(text.encode()).hexdigest()[:12]}"


round1 = [json.loads(l) for l in open(ROUND1_GEN, encoding="utf-8")] if os.path.exists(ROUND1_GEN) else []
round1_ids = {r["id"] for r in round1}
print(f"round1: {len(round1)} problems")

## Model (vLLM) + agent loop

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL, max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.92, max_num_seqs=MAX_NUM_SEQS,
          limit_mm_per_prompt={"image": 0, "video": 0}, enable_prefix_caching=True, seed=SEED)
tok = llm.get_tokenizer()
WORKERS = (os.cpu_count() or 4) * 2


def run_episodes(episodes, temperature, context_limit, gen_limit, turn_limit, label=""):
    # Each turn generates all active episodes in one batch; tools run in parallel. Shared prefixes hit the prefix cache.
    active = [e for e in episodes if not e.done]
    while active:
        prompts, params, batch = [], [], []
        for e in active:
            p = tok.apply_chat_template(e.messages, tools=e.tools, tokenize=False, add_generation_prompt=True)
            budget = min(turn_limit, context_limit - len(tok(p, add_special_tokens=False).input_ids),
                         gen_limit - e.tokens)
            if budget < 256:
                e.fail("context")
                continue
            prompts.append(p)
            params.append(SamplingParams(temperature=temperature, top_p=0.95, top_k=20, max_tokens=budget,
                                         seed=SEED + 7919 * e.rollout + e.turns))
            batch.append(e)
        if not batch:
            break
        print(f"{label} turn {batch[0].turns + 1}: {len(batch)} active episodes")
        outs = llm.generate(prompts, params, use_tqdm=True)   # progress within the turn
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            list(ex.map(lambda eo: eo[0].consume(eo[1].outputs[0].text, eo[1].outputs[0].finish_reason,
                                                  len(eo[1].outputs[0].token_ids)), zip(batch, outs)))
        active = [e for e in batch if not e.done]
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        results = list(ex.map(A.grade_episode, episodes))
    for e, (ok, detail) in zip(episodes, results):
        e.ok, e.detail = ok, str(detail)[:300]
        e.sandbox.close()
    return episodes


# Smoke test: one problem with the Python tool
e = run_episodes([A.Episode({"id": "t", "type": "math", "prompt": "Compute 123456789 * 987654321.",
                             "answer": "121932631112635269"}, max_turns=4)], 0.6, 8192, 8192, 8192)[0]
print(e.ok, e.failed, e.turns, e.n_tool_calls, e.tokens)
print(e.messages[-1]["content"][-300:])

## Agentic coding tasks (KodCode)

**Eval set of 100 tasks:**
- Split off deterministically by a hash of the question id.
- No problem seen in round 1 goes into this set.
- Saved to Drive as `eval/tasks/agent_code.jsonl`.

**RFT pool:** problems disjoint from both the eval set and round 1.

In [ ]:
from datasets import load_dataset

held_out = lambda qid: int(hashlib.md5(qid.encode()).hexdigest()[:2], 16) < 8   # ~3%
AGENT_EVAL_PATH = f"{EVAL_DIR}/tasks/agent_code.jsonl"
agent_eval = [json.loads(l) for l in open(AGENT_EVAL_PATH, encoding="utf-8")] if os.path.exists(AGENT_EVAL_PATH) else []
code_pool = []
stream = load_dataset("KodCode/KodCode-V1", split="train", streaming=True).shuffle(seed=SEED + 1, buffer_size=20_000)
for r in stream:
    if (r["style"] != "instruct" or r["gpt_difficulty"] not in ("medium", "hard")
            or not r["test_info"] or "from solution import" not in r["test"]):
        continue
    stub = A.make_stub(r["test_info"])
    if not stub or uid("code", r["question_id"]) in round1_ids:
        continue
    task = {"id": uid("agent", r["question_id"]), "type": "agent_code", "prompt": r["question"],
            "tests": r["test"], "stub": stub}
    if held_out(r["question_id"]):
        if len(agent_eval) < AGENT_EVAL_N and task["id"] not in {t["id"] for t in agent_eval}:
            agent_eval.append(task)
    elif len(code_pool) < POOL["code"]:
        code_pool.append(task)
    if len(agent_eval) >= AGENT_EVAL_N and len(code_pool) >= POOL["code"]:
        break
if not os.path.exists(AGENT_EVAL_PATH):
    with open(AGENT_EVAL_PATH, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(t, ensure_ascii=False) + "\n" for t in agent_eval)
print(f"agent eval set: {len(agent_eval)}, code pool: {len(code_pool)}")
print(agent_eval[0]["stub"])

## A) Baseline eval (with tools)

Source names: `math500+py`, `aime2025+py`, `gpqa_physics+py`, `agent_code`.
- The effect of the Python tool **on its own** shows when these results are compared with the no-tools baseline (`base-colab-fp8`).
- rft2 goes through the same eval in `04_sft`.

In [ ]:
def max_turns(task):
    return MAX_TURNS["code" if task["type"] == "agent_code" else "python"]


def summarize(episodes, path):
    # write_summary only uses n/correct/tokens/seconds; tool calls and end reasons are printed here
    stats = {}
    with open(path, "w", encoding="utf-8") as f:
        for e in episodes:
            src = e.task["_source"]
            st = stats.setdefault(src, {"n": 0, "correct": 0, "tokens": 0, "seconds": 0.0, "tools": 0, "failed": collections.Counter()})
            st["n"] += 1
            st["correct"] += int(e.ok)
            st["tokens"] += e.tokens
            st["tools"] += e.n_tool_calls
            st["failed"][e.failed] += 1
            f.write(json.dumps({"id": e.task["id"], "source": src, "ok": e.ok, "detail": e.detail, "tokens": e.tokens,
                                "turns": e.turns, "tool_calls": e.n_tool_calls, "failed": e.failed},
                               ensure_ascii=False) + "\n")
    for src, st in stats.items():
        print(f"  {src:<18} tool calls/episode {st['tools'] / st['n']:.1f} | end reason: { {(k or 'finished'): v for k, v in st['failed'].items()} }")
    return stats


if RUN_EVAL:
    eval_tasks = R.load_tasks([f"{EVAL_DIR}/tasks/{n}.jsonl" for n in EVAL_SETS], 0)
    for t in eval_tasks:
        t["_source"] += "+py"
    for t in agent_eval:
        t["_source"] = "agent_code"
    episodes = [A.Episode(t, rollout=k, max_turns=max_turns(t)) for t in eval_tasks + agent_eval for k in range(EVAL_SAMPLES)]
    t0 = time.time()
    run_episodes(episodes, EVAL_TEMPERATURE, EVAL_CONTEXT, EVAL_GEN_TOKENS, EVAL_TURN_TOKENS, "eval")
    elapsed = time.time() - t0
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    stats = summarize(episodes, f"{EVAL_DIR}/results/{EVAL_LABEL}_{stamp}.jsonl")
    total = sum(st["n"] for st in stats.values())
    for st in stats.values():
        st["seconds"] = elapsed * st["n"] / total
    R.write_summary(f"{EVAL_DIR}/results", stamp, EVAL_LABEL, MODEL, stats)
    print(f"Time: {elapsed / 60:.1f} min ≈ {elapsed / 3600 * CU_PER_HOUR:.1f} CU")

## B) RFT pool + generation (resumable)

**Pool:**
- **py_hard:** math/physics problems solved at most 2/4 in round 1. This is where the Python tool can help most.
- **py_new:** new math/physics problems.
- **code:** agentic coding tasks.

In [ ]:
rng = random.Random(SEED)
hard = [r["task"] for r in round1 if r["task"]["type"] in ("math", "numeric")
        and sum(s["ok"] for s in r["samples"]) <= 2]
rng.shuffle(hard)
py_pool = hard[:POOL["py_hard"]]

ds = load_dataset("agentica-org/DeepScaleR-Preview-Dataset", split="train").shuffle(seed=SEED + 1)
new_math = [{"id": uid("math", r["problem"]), "type": "math", "prompt": r["problem"], "answer": r["answer"]}
            for r in ds.select(range(3000))]
ds = load_dataset("TIGER-Lab/WebInstruct-verified", split="train")
ds = ds.filter(lambda r: r["category"] == "Physics" and r["answer_type"] in ("Float", "Integer", "Percentage")
               and r["difficulty"] in ("Senior High School", "University", "PhD")).shuffle(seed=SEED + 1)
new_phys = [{"id": uid("phys", r["question"]), "type": "numeric", "prompt": r["question"], "answer": r["answer"]}
            for r in ds.select(range(3000))]
fresh = [t for t in new_math + new_phys if t["id"] not in round1_ids]
rng.shuffle(fresh)
py_pool += fresh[:POOL["py_new"]]

pool = [{**t, "id": "py:" + t["id"]} for t in py_pool] + code_pool
rng.shuffle(pool)
print(f"pool: {len(pool)} tasks ({len(py_pool)} python, {len(code_pool)} code)")

EP_PATH = f"{OUT_DIR}/episodes.jsonl"
done = set()
if os.path.exists(EP_PATH):
    done = {json.loads(l)["id"] for l in open(EP_PATH, encoding="utf-8") if l.strip()}
todo = [t for t in pool if t["id"] not in done]
print(f"Done: {len(done)}, remaining: {len(todo)}")

start, chunk_hours = time.time(), 0.0
for i in range(0, len(todo), CHUNK):
    if (time.time() - start) / 3600 + chunk_hours > MAX_GEN_HOURS:
        print("Time budget reached, stopping.")
        break
    chunk = todo[i:i + CHUNK]
    t0 = time.time()
    eps = [A.Episode(t, rollout=k, max_turns=max_turns(t)) for t in chunk for k in range(N_SAMPLES)]
    run_episodes(eps, GEN_TEMPERATURE, GEN_CONTEXT, GEN_GEN_TOKENS, GEN_TURN_TOKENS, f"[{i + len(chunk)}/{len(todo)}]")
    by_task = collections.defaultdict(list)
    for e in eps:
        by_task[e.task["id"]].append(e)
    with open(EP_PATH, "a", encoding="utf-8") as f:
        for t in chunk:
            rolls = [{"ok": e.ok, "tokens": e.tokens, "turns": e.turns, "tool_calls": e.n_tool_calls, "failed": e.failed,
                      "messages": e.messages if e.ok else None} for e in by_task[t["id"]]]
            f.write(json.dumps({"id": t["id"], "task": t, "rollouts": rolls}, ensure_ascii=False) + "\n")
    chunk_hours = (time.time() - t0) / 3600
    acc = sum(e.ok for e in eps) / len(eps)
    hours = (time.time() - start) / 3600
    print(f"== {i + len(chunk)}/{len(todo)} | accuracy {acc:.0%} | chunk {chunk_hours * 60:.0f} min | "
          f"{hours:.2f} h ≈ {hours * CU_PER_HOUR:.1f} CU")

## B) Selection → SFT data

- Only correct and complete episodes are used. For code, that means the tests pass after the test file is restored.
- Among the episodes that fit the training limit of **10240 tokens**, the **shortest** are picked:
  - at most 2 episodes from partially solved tasks,
  - one episode from 50% of the tasks solved 4/4.
- Output: `agent_sft.jsonl`. `04_sft` merges it with the round-1 data and trains on both.

In [ ]:
TRAIN_MAX = 10240
EASY_KEEP, MAX_PER_TASK = 0.5, 2

rng = random.Random(SEED)
stats = collections.defaultdict(collections.Counter)
examples = []
for line in open(EP_PATH, encoding="utf-8"):
    row = json.loads(line)
    t, rolls = row["task"], row["rollouts"]
    src = "code" if t["type"] == "agent_code" else "py_" + t["type"]
    tools = A.CODE_TOOLS if t["type"] == "agent_code" else [A.PYTHON_TOOL]
    good = []
    for r in rolls:
        if r["ok"]:
            text = tok.apply_chat_template(r["messages"], tools=tools, tokenize=False)
            n = len(tok(text, add_special_tokens=False).input_ids)
            if n <= TRAIN_MAX:
                good.append((n, r))
    st = stats[src]
    st["tasks"] += 1
    st["correct_eps"] += sum(r["ok"] for r in rolls)
    st["total_eps"] += len(rolls)
    st["solved_fitting"] += bool(good)
    if not good:
        continue
    good.sort(key=lambda x: x[0])
    if len(good) == len(rolls):
        if rng.random() > EASY_KEEP:
            continue
        good = good[:1]
    else:
        good = good[:MAX_PER_TASK]
    for n, r in good:
        examples.append({"messages": r["messages"], "tools": tools, "source": src, "n_tokens": n})
        st["kept"] += 1
        st["tokens"] += n
        st["used_tools"] += r["tool_calls"] > 0

rng.shuffle(examples)
with open(f"{OUT_DIR}/agent_sft.jsonl", "w", encoding="utf-8") as f:
    f.writelines(json.dumps(ex, ensure_ascii=False) + "\n" for ex in examples)

print(f"{'source':<11} {'tasks':>6} {'solved':>8} {'correct%':>8} {'kept':>8} {'avg tok':>8} {'w/ tools':>8}")
for src, st in stats.items():
    k = max(1, st["kept"])
    print(f"{src:<11} {st['tasks']:>6} {st['solved_fitting']:>8} {st['correct_eps'] / st['total_eps']:>8.1%} "
          f"{st['kept']:>8} {st['tokens'] / k:>8.0f} {st['used_tools'] / k:>8.0%}")
print(f"\nTotal: {len(examples)} -> {OUT_DIR}/agent_sft.jsonl")

In [ ]:
# Check: a rendered code example (multi-turn: thinking + tool call + tool_response)
ex = next((e for e in examples if e["source"] == "code"), None)
if ex:
    print(tok.apply_chat_template(ex["messages"], tools=ex["tools"], tokenize=False)[-2500:])

## Finish: stop spending CU

In [ ]:
from google.colab import runtime
runtime.unassign()